In [1]:
import os
import sys

project_root = os.path.abspath('..')
sys.path.append(project_root)

# Somehow this is needed for the imports to work
# Else it takes infinite time to import the modules
from gevent import monkey
monkey.patch_all()

# Now import the required modules
from mu_bench.metrics_exporter.prometheus_exporter import PrometheusExporter
from mu_bench.request_generator.edge_request_generator import EdgeRequestGenerator
from mu_bench.experiment_manager import EdgeExperimentManager

In [2]:
config = {
    "base_host": "172.23.0.5",
    "apps": [
        {
            "name": "app-edge",  # Different name creates different hash
            "port": 31115,
            "traffic_weight": 1,
            "wait_time": {"min": 0.7, "max": 1.5}
        },
        {
            "name": "app-edge-2",  # Different name = different behavior
            "port": 31116,
            "traffic_weight": 0.9,
            "wait_time": {"min": 0.8, "max": 2.5}
        },
    ],
    "edge_sites": {
        "edge-site-1": {
            "pattern": "urban_peak",
            "description": "Urban site with business peaks",
            "traffic_weight": 1.0
        },
        "edge-site-2": {
            "pattern": "suburban_steady", 
            "description": "Suburban site with steady traffic",
            "traffic_weight": 0.8
        }
    }
}

In [3]:
prometheus_logger = PrometheusExporter(
    prometheus_url="http://localhost:9090",
    target_namespaces=["app-edge"],
    collection_interval_minutes=5,
    metrics_step="30s",
    output_dir="metrics",
    verbose=True,
    create_output_dir=False
)

# Create traffic generator
traffic_generator = EdgeRequestGenerator()

# Create experiment manager
experiment_manager = EdgeExperimentManager(
    prometheus_exporter=prometheus_logger,
    traffic_generator=traffic_generator,
    experiment_name=None,
    output_base_dir="../DATA"
)

# Run experiment

try:
    success = experiment_manager.run_experiment(
        edge_config=config,
        duration_hours=1,        # Real time: 7 hours
        simulated_hours=24,      # Simulated time: 24 hours
        max_users=40,
        spawn_rate=3
    )
except Exception as e:
    print("\n🚫 Experiment interrupted by user")
finally:
    if success:
        print(f"\n🎉 Experiment completed! Results: {experiment_manager.output_directory}")
    else:
        print("\n💥 Experiment failed!")


🚀 EXPERIMENT: edge_experiment_2025_06_02_14_31
📋 CONFIGURATION
Duration:       1h real → 24h simulated
Host:           172.23.0.5
Output:         edge_experiment_2025_06_02_14_31
Metrics:        Every 5min

🔌 APPLICATIONS (2)
  app-edge     Port 31115  Weight 1
  app-edge-2   Port 31116  Weight 0.9

🌍 EDGE SITES (2)
  edge-site-1     urban_peak      Weight 1.0
  edge-site-2     suburban_steady Weight 0.8

📈 ENDPOINTS: 4 total
Starting metrics collection every 5 minutes
Metrics step interval: 30s
Output directory: ../DATA/edge_experiment_2025_06_02_14_31/metrics
Using 30s step intervals for the past 5 minutes
Metrics collection started
Metrics collection completed at 2025-06-02T11:31:39.665745+00:00


Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated                                                                         0     0(0.00%) |      0       0       0      0 |    0.00        0.00



GlobalUserCountShape: Simulated time reset to 0.00s.


Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
GET      31115:/s0-edge-site-1                                                              2     0(0.00%) |    222     129     315    130 |    0.00        0.00
GET      31115:/s0-edge-site-2                                                              4     0(0.00%) |     71      51      91     70 |    0.00        0.00
GET      31116:/s0-edge-site-1                                                              1     0(0.00%) |    484     484     484    484 |    0.00        0.00
GET      31116:/s0-edge-site-2                                                              1     0(0.00%) |    609     609     609    609 |    0.00        0.00
--------|-------------------------

Using 30s step intervals for the past 5 minutes
Metrics collection completed at 2025-06-02T11:36:39.689484+00:00


Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
GET      31115:/s0-edge-site-1                                                            995     0(0.00%) |    227      15    3390    160 |    4.00        0.00
GET      31115:/s0-edge-site-2                                                            970     0(0.00%) |    242      15    1762    170 |    3.00        0.00
GET      31116:/s0-edge-site-1                                                            344     0(0.00%) |   1906     477    5888   1700 |    0.70        0.00
GET      31116:/s0-edge-site-2                                                            323     3(0.93%) |   1986     106    7282   1800 |    1.20        0.00
--------|-------------------------

... (5 outputs left out)

Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
GET      31115:/s0-edge-site-1                                                           5635     1(0.02%) |    233      14    3390    170 |    7.40        0.00
GET      31115:/s0-edge-site-2                                                           5405     0(0.00%) |    234      15    1833    170 |    5.70        0.00
GET      31116:/s0-edge-site-1                                                           1591    16(1.01%) |   2705      37   20049   2400 |    2.50        0.00
GET      31116:/s0-edge-site-2                                                           1467    17(1.16%) |   2618      18   11545   2300 |    1.30        0.00
--------|-------------------------

Stopping metrics collection...
Stopping metrics collection...
Metrics collection stopped
📊 Metrics collection completed

💥 Experiment failed!


Failed to log request: I/O operation on closed file.


Failed to log request: I/O operation on closed file.
